In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo
import torch

In [13]:
now = datetime.now(ZoneInfo("America/Los_Angeles"))
now.strftime("%Y-%m-%d-%H-%M-%S-%s")

'2026-06-29-06-43-55-1782740635'

In [30]:

p = torch.tensor([
    [0.0, 0.2, 0.5, 0.78, 0.99, 1.0],
    [0.0, 0.2, 0.5, 0.78, 0.99, 1.0]
])

In [33]:
pp = torch.cumsum(p, dim=-1)
pp

tensor([[0.0000, 0.2000, 0.7000, 1.4800, 2.4700, 3.4700],
        [0.0000, 0.2000, 0.7000, 1.4800, 2.4700, 3.4700]])

In [ ]:
# torch.bucketize(torch.tensor([0.5, 0.9])[:, None], pp, out_int32=True).item()


RuntimeError: boundaries tensor must be 1 dimension, but got dim(2)

In [49]:
def bucketize(query_1d: torch.Tensor, boundaries_2d: torch.Tensor) -> torch.Tensor:
    # query is of size 1xb, boundaries of size b x vocab
    # result will be 1xb, one index from boundaries for each
    m = (query_1d <= boundaries_2d) * 1 #b x vocab
    # print(query_1d)
    # print(boundaries_2d)
    # print(m)

    return boundaries_2d.shape[-1] - torch.sum(m, dim=-1)


In [50]:
bucketize(torch.tensor([0.5, 0.9])[:, None], pp)

tensor([2, 3])

In [46]:
torch.rand(pp.shape[0])

tensor([0.3330, 0.4057])

In [27]:
#emb.shape = [seq_length, token_dim]

#emb[pos, 2i]   = sin(pos / 10000^(2i/token_dim))
#emb[pos, 2i+1] = cos(pos / 10000^(2i/token_dim))

token_dim = 6
seq_length = 4

x_axis = torch.arange(seq_length, dtype=torch.int)
y_axis_even = torch.arange(0, token_dim, 2, dtype=torch.int)
y_axis_odd = torch.arange(1, token_dim, 2, dtype=torch.int)

positional_emb = torch.empty(seq_length, token_dim, dtype=torch.float16)


# torch.cos(torch.tensor([1, 2, 3]))
seq_positions = torch.arange(seq_length)[:, None] #4, 1
# print(seq_positions)
even_powers = torch.sin(10000 ** (torch.arange(0, token_dim, 2) / token_dim)) #1, 3
odd_powers = torch.cos(10000 ** (torch.arange(0, token_dim, 2) / token_dim)) #1, 3
# print(even_powers)
# print(seq_positions / even_powers)

positional_emb[x_axis[:, None], y_axis_even[None, :]] = (seq_positions / even_powers).to(torch.float16)
positional_emb[x_axis[:, None], y_axis_odd[None, :]] = (seq_positions / odd_powers).to(torch.float16)

print(positional_emb)



tensor([[ 0.0000,  0.0000,  0.0000, -0.0000, -0.0000,  0.0000],
        [ 1.1885,  1.8506,  2.3145, -1.1084, -1.3984,  1.4307],
        [ 2.3770,  3.7012,  4.6289, -2.2168, -2.7969,  2.8613],
        [ 3.5645,  5.5508,  6.9414, -3.3262, -4.1953,  4.2930]],
       dtype=torch.float16)
